In [ ]:
import os
import random
import sys
import warnings

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append('..')
from src.models.dlinear import create_dlinear_adapter
from src.utils.metrics import nwrmsle

In [ ]:
warnings.filterwarnings('ignore')

In [2]:
RAW_DATA_DIR = r"..\data\raw_data"
PREP_DATA_DIR = r"..\data\prepared_data"

In [3]:
train = pd.read_parquet(os.path.join(PREP_DATA_DIR, "train_filled_last3m.parquet"))
items = pd.read_csv(os.path.join(RAW_DATA_DIR, "items.csv"))
stores = pd.read_csv(os.path.join(RAW_DATA_DIR, "stores.csv"))

In [4]:
item_weights = items.set_index('item_nbr')['perishable'].to_dict()
item_weights = {k: 1.25 if v == 1 else 1.0 for k, v in item_weights.items()}

train_end_date = pd.Timestamp('2017-07-29')
val_end_date = pd.Timestamp('2017-08-14')

train_data = train[train['date'] <= train_end_date].copy()
val_data = train[(train['date'] > train_end_date) & (train['date'] <= val_end_date)].copy()
val_data['weight'] = val_data['item_nbr'].map(item_weights).fillna(1.0)

print(f"Train: {len(train_data)} записей")
print(f"Val: {len(val_data)} записей")
print(f"Уникальных пар в train: {train_data.groupby(['store_nbr', 'item_nbr']).ngroups}")

Train: 106124807 записей
Val: 2538176 записей
Уникальных пар в train: 174064


In [ ]:
pair_counts = train_data.groupby(['store_nbr', 'item_nbr']).size().sort_values(ascending=False)
sample_pairs = pair_counts.head(50).index.tolist()
print(f"Выбрано пар: {len(sample_pairs)}")

Выбрано пар: 25


In [ ]:
predictions = []

for idx, (store, item) in enumerate(sample_pairs):
    if idx % 10 == 0:
        print(f"Прогресс: {idx}/{len(sample_pairs)}")
    
    # Берем данные для этой пары
    pair_data = train_data[
        (train_data['store_nbr'] == store) & 
        (train_data['item_nbr'] == item)
    ].sort_values('date')
    
    if len(pair_data) < 70:
        print(f"  Пропускаем: слишком мало данных ({len(pair_data)})")
        continue
    
    # Данные для обучения (все кроме последних 16 дней)
    train_pair = pair_data.iloc[:-16].copy()
    train_pair = train_pair.rename(columns={'unit_sales': 'value'})
    
    # Последние данные для контекста
    last_data = pair_data[['date', 'unit_sales']].rename(
        columns={'unit_sales': 'value'}
    )
    
    # Обучаем
    dlinear_copy = create_dlinear_adapter(seq_len=60, pred_len=16, epochs=20)
    dlinear_copy.fit(train_data=train_pair)
    
    # Прогнозируем
    forecast = dlinear_copy.predict(16, last_data)
    
    # Сохраняем прогнозы
    last_date = pair_data['date'].max()
    for i, pred in enumerate(forecast):
        predictions.append({
            'store_nbr': store,
            'item_nbr': item,
            'date': last_date + pd.Timedelta(days=i+1),
            'predicted': max(0, pred)
        })


--- Пара 1/25: магазин 8, товар 581078 ---
✅ DLinear инициализирован (seq_len=60, pred_len=16) на cpu

--- Пара 2/25: магазин 49, товар 168927 ---
✅ DLinear инициализирован (seq_len=60, pred_len=16) на cpu

--- Пара 3/25: магазин 44, товар 838407 ---
✅ DLinear инициализирован (seq_len=60, pred_len=16) на cpu

--- Пара 4/25: магазин 44, товар 838408 ---
✅ DLinear инициализирован (seq_len=60, pred_len=16) на cpu

--- Пара 5/25: магазин 10, товар 587157 ---
✅ DLinear инициализирован (seq_len=60, pred_len=16) на cpu

--- Пара 6/25: магазин 49, товар 215896 ---
✅ DLinear инициализирован (seq_len=60, pred_len=16) на cpu

--- Пара 7/25: магазин 51, товар 838409 ---
✅ DLinear инициализирован (seq_len=60, pred_len=16) на cpu

--- Пара 8/25: магазин 51, товар 838408 ---
✅ DLinear инициализирован (seq_len=60, pred_len=16) на cpu

--- Пара 9/25: магазин 49, товар 213652 ---
✅ DLinear инициализирован (seq_len=60, pred_len=16) на cpu

--- Пара 10/25: магазин 51, товар 838216 ---
✅ DLinear инициализ

In [ ]:
pred_df = pd.DataFrame(predictions)
print(f"\nПолучено прогнозов: {len(pred_df)}")

# Оцениваем
val_with_pred = val_data.merge(
    pred_df[['store_nbr', 'item_nbr', 'date', 'predicted']],
    on=['store_nbr', 'item_nbr', 'date'],
    how='inner'
)

In [ ]:
score = nwrmsle(
    val_with_pred['unit_sales'].values,
    val_with_pred['predicted'].values,
    val_with_pred['weight'].values
)
print(f"DLinear NWRMSLE: {score:.6f}")


Получено прогнозов: 400

📊 DLinear NWRMSLE: 0.519429


In [1]:
# Настройка стиля для графиков
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (15, 8)
plt.rcParams['font.size'] = 12

# Выберем случайную пару из тех, что есть в результатах
available_pairs = val_with_pred[['store_nbr', 'item_nbr']].drop_duplicates().values.tolist()
if available_pairs:
    example_store, example_item = random.choice(available_pairs)
    
    example_data = val_with_pred[
        (val_with_pred['store_nbr'] == example_store) & 
        (val_with_pred['item_nbr'] == example_item)
    ].sort_values('date')
    
    if len(example_data) > 0:
        fig, ax = plt.subplots(figsize=(15, 6))
        
        # Основные линии
        ax.plot(example_data['date'], example_data['unit_sales'], 
                'b-', label='Факт', linewidth=2.5, marker='o', markersize=6, alpha=0.8)
        ax.plot(example_data['date'], example_data['predicted'], 
                'r--', label='Прогноз DLinear', linewidth=2.5, marker='s', markersize=6, alpha=0.8)
        
        # Заполнение области между кривыми (ошибка)
        ax.fill_between(example_data['date'], 
                        example_data['unit_sales'], 
                        example_data['predicted'],
                        alpha=0.2, color='gray', label='Ошибка')
        
        # Настройка графика
        ax.set_title(f'Магазин {example_store}, Товар {example_item}\nDLinear прогноз (NWRMSLE: {score:.4f})', 
                    fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('Дата', fontsize=13)
        ax.set_ylabel('Продажи (unit sales)', fontsize=13)
        ax.legend(loc='upper right', fontsize=11)
        ax.grid(True, alpha=0.3, linestyle='--')
        
        # Форматирование дат
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d.%m.%Y'))
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
        plt.xticks(rotation=45, ha='right')
        
        plt.tight_layout()
        plt.show()

NameError: name 'plt' is not defined